# <font color="#418FDE" size="6.5" uppercase>**Distances Kernels**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Compute and interpret pairwise distance matrices for common metrics. 
- Use kernel functions to represent similarity for linear and nonlinear modeling methods. 
- Decide when precomputed distances or kernels are appropriate in Scikit-Learn workflows. 


## **1. Common Distance Metrics**

### **1.1. Euclidean Distance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_01_01.jpg?v=1783837964" width="250">



>* Euclidean distance measures straight-line feature differences.
>* Distance matrices reveal close and distant pairs.

>* Feature scale strongly affects Euclidean distances.
>* Best for comparable continuous features.

>* Distance matrices reveal clusters and outliers.
>* Euclidean distance is useful but limited.



In [ ]:
#@title Python Code - Euclidean Distance

# Euclidean distance compares straight-line separation.
# Small examples make pairwise distances intuitive.
# Feature scale strongly affects distance values.

import numpy as np
import matplotlib.pyplot as plt

# Create three simple two-feature observations.
points = np.array([[1.0, 1.0], [2.0, 1.0], [4.0, 3.0]])
labels = ["A", "B", "C"]

# Compute Euclidean pairwise distances manually.
diff = points[:, None, :] - points[None, :, :]
distances = np.sqrt((diff ** 2).sum(axis=2))

# Check the expected square matrix shape.
assert distances.shape == (3, 3)
print("Points:", dict(zip(labels, points.tolist())))
print("A to B distance:", round(distances[0, 1], 2))

# Show one farther pair for comparison.
print("A to C distance:", round(distances[0, 2], 2))
print("Distance matrix:\n", np.round(distances, 2))

# Plot the points and connecting segments.
plt.figure(figsize=(5, 4))
plt.scatter(points[:, 0], points[:, 1], s=80)

# Label each point clearly.
for name, xy in zip(labels, points):
    plt.text(xy[0] + 0.05, xy[1] + 0.05, name)

# Draw lines from point A.
plt.plot(points[[0, 1], 0], points[[0, 1], 1])
plt.plot(points[[0, 2], 0], points[[0, 2], 1])

# Finish the visual explanation.
plt.title("Euclidean distance between simple points")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

# Keep the plot readable.
plt.axis("equal")
plt.grid(True, alpha=0.3)
plt.show()



### **1.2. Metric Distance Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_01_02.jpg?v=1783837967" width="250">



>* Metrics consistently measure observation differences.
>* Metric rules ensure stable, interpretable comparisons.

>* Matrices store distances between every observation pair.
>* Interpret values by scale, context, and clusters.

>* Choose metrics that match data meaning.
>* Metric choice shapes similarity and interpretation.



In [ ]:
#@title Python Code - Metric Distance Basics

# This script introduces basic distance metrics.
# Small examples make pairwise distances easy.
# Results show symmetry and zero diagonals.

import numpy as np

# Create three simple two feature observations.
points = np.array([[0, 0], [3, 4], [3, 0]], dtype=float)
labels = ["A", "B", "C"]

# Define Euclidean distance with numpy.
def euclidean_distance(a, b):

    return np.sqrt(np.sum((a - b) ** 2))

# Define Manhattan distance with numpy.
def manhattan_distance(a, b):

    return np.sum(np.abs(a - b))

# Build a pairwise distance matrix.
def pairwise_matrix(data, metric_function):

    n_rows = data.shape[0]
    matrix = np.zeros((n_rows, n_rows))
    for i in range(n_rows):
        for j in range(n_rows):

            matrix[i, j] = metric_function(data[i], data[j])
    return matrix

# Validate the tiny dataset shape.
assert points.ndim == 2 and points.shape[0] == 3

# Compute two common distance matrices.
euclidean_matrix = pairwise_matrix(points, euclidean_distance)
manhattan_matrix = pairwise_matrix(points, manhattan_distance)

# Print the observations first.
print("Points:", dict(zip(labels, points.tolist())))

# Print rounded Euclidean distances.
print("Euclidean matrix:\n", np.round(euclidean_matrix, 2))

# Print rounded Manhattan distances.
print("Manhattan matrix:\n", np.round(manhattan_matrix, 2))

# Check key metric distance properties.
print("Zero diagonal:", np.allclose(np.diag(euclidean_matrix), 0.0))
print("Symmetric:", np.allclose(euclidean_matrix, euclidean_matrix.T))

# Interpret one pair under both metrics.
print("A to B Euclidean:", round(euclidean_matrix[0, 1], 2))
print("A to B Manhattan:", round(manhattan_matrix[0, 1], 2))



### **1.3. Metric Distance Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_01_03.jpg?v=1783837971" width="250">



>* Distances summarize observation differences in a matrix.
>* Good metrics are consistent and interpretable.

>* Distance meaning depends on metric and scale.
>* Standardize features for fair similarity comparisons.

>* Distance metrics define similarity differently.
>* Interpret matrices using metric and data meaning.



In [ ]:
#@title Python Code - Metric Distance Basics

# Pairwise distances compare observations numerically.
# Different metrics can change nearest neighbors.
# Small examples make matrix patterns easier.

import numpy as np
import pandas as pd

# Create tiny house feature data.
houses = pd.DataFrame({
    "size_sqft": [900, 1100, 2000],
    "age_years": [30, 28, 5],
    "transit_miles": [0.5, 1.0, 8.0],
}, index=["House A", "House B", "House C"])

# Convert table to numeric matrix.
X = houses.to_numpy(dtype=float)

# Define Euclidean distance function.
def euclidean_matrix(data):
    diffs = data[:, None, :] - data[None, :, :]

    squared = diffs ** 2
    summed = squared.sum(axis=2)
    return np.sqrt(summed)

# Define Manhattan distance function.
def manhattan_matrix(data):
    diffs = np.abs(data[:, None, :] - data[None, :, :])

    return diffs.sum(axis=2)

# Standardize features for fair comparison.
means = X.mean(axis=0)
stds = X.std(axis=0)

safe_stds = np.where(stds == 0, 1.0, stds)
X_scaled = (X - means) / safe_stds

# Compute three pairwise distance matrices.
euclid_raw = euclidean_matrix(X)
manhattan_raw = manhattan_matrix(X)

euclid_scaled = euclidean_matrix(X_scaled)

# Wrap matrices with readable labels.
euclid_df = pd.DataFrame(
    np.round(euclid_raw, 2),
    index=houses.index,
    columns=houses.index,
)

manhattan_df = pd.DataFrame(
    np.round(manhattan_raw, 2),
    index=houses.index,
    columns=houses.index,
)

euclid_scaled_df = pd.DataFrame(
    np.round(euclid_scaled, 2),
    index=houses.index,
    columns=houses.index,
)

# Find nearest neighbor excluding self.
def nearest_neighbor(distance_df, name):
    row = distance_df.loc[name].copy()

    row.loc[name] = np.inf
    return row.idxmin(), row.min()

# Summarize one observation across metrics.
nn_raw, d_raw = nearest_neighbor(euclid_df, "House A")
nn_scaled, d_scaled = nearest_neighbor(euclid_scaled_df, "House A")

# Print compact teaching output.
print("House features:")
print(houses)
print("\nEuclidean distances, raw features:")
print(euclid_df)
print("\nManhattan distances, raw features:")
print(manhattan_df)
print("\nEuclidean distances, standardized features:")
print(euclid_scaled_df)
print(f"\nNearest to House A, raw Euclidean: {nn_raw} ({d_raw:.2f})")
print(f"Nearest to House A, scaled Euclidean: {nn_scaled} ({d_scaled:.2f})")



## **2. Similarity Kernels**

### **2.1. Kernel Similarity Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_02_01.jpg?v=1783837950" width="250">



>* Kernels turn comparisons into similarity scores.
>* They help models learn from relationships.

>* Linear kernels capture straight-line feature similarity.
>* Nonlinear kernels reveal complex patterns implicitly.

>* Kernel choice determines learned similarity patterns.
>* Different applications need different similarity assumptions.



In [ ]:
#@title Python Code - Kernel Similarity Basics

# Kernels turn comparisons into similarity scores.
# This example contrasts linear and radial kernels.
# Small matrices make similarity ideas easy.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny two-feature observations.
X = np.array([
    [1.0, 1.0],
    [2.0, 2.0],
    [1.0, -1.0],
    [-1.5, -1.0],
])

# Name points for readable output.
labels = ["A", "B", "C", "D"]

# Compute a linear similarity matrix.
linear_kernel = X @ X.T

# Compute Euclidean distances first.
diff = X[:, None, :] - X[None, :, :]
dist_sq = np.sum(diff ** 2, axis=2)

# Convert distances into radial similarity.
gamma = 0.5
rbf_kernel = np.exp(-gamma * dist_sq)

# Check matrix shapes defensively.
assert linear_kernel.shape == (4, 4)
assert rbf_kernel.shape == (4, 4)

# Print a compact interpretation.
print("Points:", labels)
print("Linear A-B, A-C:", round(linear_kernel[0, 1], 2), round(linear_kernel[0, 2], 2))
print("RBF A-B, A-C:", round(rbf_kernel[0, 1], 3), round(rbf_kernel[0, 2], 3))
print("RBF diagonal:", np.round(np.diag(rbf_kernel), 3))
print("Higher values mean stronger similarity.")

# Plot both kernel matrices side by side.
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

# Show the linear kernel image.
axes[0].imshow(linear_kernel, cmap="Blues")
axes[0].set_title("Linear kernel")
axes[0].set_xticks(range(4), labels)
axes[0].set_yticks(range(4), labels)

# Show the radial kernel image.
axes[1].imshow(rbf_kernel, cmap="Oranges", vmin=0, vmax=1)
axes[1].set_title("RBF kernel")
axes[1].set_xticks(range(4), labels)
axes[1].set_yticks(range(4), labels)

# Tight layout keeps labels readable.
plt.tight_layout()
plt.show()



### **2.2. Kernel Similarity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_02_02.jpg?v=1783837947" width="250">



>* Kernels measure similarity beyond simple distance.
>* Kernel matrices guide many learning tasks.

>* Kernel choice shapes linear or nonlinear similarity.
>* Best kernels match the problem's structure.

>* Kernel similarity depends on chosen resemblance rules.
>* Good kernels need relevant features and balance.



In [ ]:
#@title Python Code - Kernel Similarity

# Kernels measure similarity between pairs.
# Different kernels emphasize different relationships.
# This example compares linear and radial kernels.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny two dimensional observations.
X = np.array([
    [0.0, 0.0],
    [1.0, 0.2],
    [0.2, 1.0],
    [2.0, 2.0],
])

# Check the expected matrix shape.
assert X.shape == (4, 2)

# Define a simple linear kernel.
def linear_kernel(A, B):
    return A @ B.T

# Define a radial basis kernel.
def rbf_kernel(A, B, gamma=1.0):
    diff = A[:, None, :] - B[None, :, :]
    sq_dist = np.sum(diff ** 2, axis=2)
    return np.exp(-gamma * sq_dist)

# Compute two similarity matrices.
K_linear = linear_kernel(X, X)
K_rbf = rbf_kernel(X, X, gamma=1.0)

# Compare one pair under both kernels.
print("Points shape:", X.shape)
print("Linear self similarity:", round(K_linear[1, 1], 3))
print("Linear pair similarity:", round(K_linear[1, 2], 3))
print("RBF self similarity:", round(K_rbf[1, 1], 3))
print("RBF pair similarity:", round(K_rbf[1, 2], 3))
print("RBF far similarity:", round(K_rbf[0, 3], 3))

# Plot both kernel similarity matrices.
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(K_linear, cmap="Blues")
axes[0].set_title("Linear kernel")
axes[0].set_xlabel("Point index")

axes[1].imshow(K_rbf, cmap="Oranges")
axes[1].set_title("RBF kernel")
axes[1].set_xlabel("Point index")

plt.tight_layout()
plt.show()



### **2.3. Kernel Similarity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_02_03.jpg?v=1783837949" width="250">



>* Kernels measure similarity between example pairs.
>* Linear aligns features; others capture nonlinear resemblance.

>* Nonlinear kernels capture curved, local similarities.
>* They reveal meaningful patterns across varied domains.

>* Kernel choice defines meaningful similarity patterns.
>* Balance interpretability, flexibility, and noise sensitivity.



In [ ]:
#@title Python Code - Kernel Similarity

# Kernels measure similarity between data points.
# Linear and nonlinear rules compare patterns differently.
# This example visualizes two common kernels.

import numpy as np
import matplotlib.pyplot as plt

# Use a deterministic seed.
rng = np.random.default_rng(7)

# Create tiny two dimensional points.
X = rng.normal(loc=0.0, scale=1.0, size=(8, 2))

# Validate the small input shape.
assert X.shape == (8, 2), "Unexpected point matrix shape."

# Compute a linear similarity matrix.
linear_kernel = X @ X.T

# Compute squared pairwise distances.
diffs = X[:, None, :] - X[None, :, :]
sq_dists = np.sum(diffs * diffs, axis=2)

# Convert distances into RBF similarity.
gamma = 0.8
rbf_kernel = np.exp(-gamma * sq_dists)

# Show one point pair comparison.
print("Linear similarity [0,1]:", round(linear_kernel[0, 1], 3))
print("RBF similarity [0,1]:", round(rbf_kernel[0, 1], 3))
print("RBF diagonal value:", round(rbf_kernel[0, 0], 3))

# Plot both kernel similarity matrices.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(8, 3.5))

# Display the linear kernel.
axes[0].imshow(linear_kernel, cmap="coolwarm")
axes[0].set_title("Linear kernel")
axes[0].set_xlabel("Point index")
axes[0].set_ylabel("Point index")

# Display the RBF kernel.
axes[1].imshow(rbf_kernel, cmap="viridis")
axes[1].set_title("RBF kernel")
axes[1].set_xlabel("Point index")
axes[1].set_ylabel("Point index")

# Tighten layout for readability.
plt.tight_layout()
plt.show()



## **3. Precomputed Similarity Inputs**

### **3.1. When Precomputed Fits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_03_01.jpg?v=1783837956" width="250">



>* Use precomputed similarities for hard-to-vectorize data.
>* Pairwise relationships can preserve crucial structure.

>* Use precomputed inputs for specialized costly similarities.
>* Direct expert comparisons can better reflect reality.

>* Best for pairwise methods on stable datasets.
>* Matrix must match samples and meaningful similarity.



### **3.2. When Precomputed Fits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_03_02.jpg?v=1783837961" width="250">



>* Use precomputed inputs for relationship-based data.
>* Best when domain similarity beats raw features.

>* Precompute costly or specialized similarities once.
>* Reuse consistent relationship matrices across analyses.

>* Use precomputed matrices with pairwise-based algorithms.
>* Best when similarity captures mixed data faithfully.



In [ ]:
#@title Python Code - When Precomputed Fits

# Pairwise relationships can be the main data.
# This example shows when precomputed fits.
# We compare objects without original features.

import numpy as np

# Build names for hard to vectorize objects.
objects = ["cat", "cut", "cute", "at"]

# Define a tiny edit distance function.
def edit_distance(a, b):
    rows, cols = len(a) + 1, len(b) + 1
    table = np.zeros((rows, cols), dtype=int)

    table[:, 0] = np.arange(rows)
    table[0, :] = np.arange(cols)
    for i, char_a in enumerate(a, start=1):

        for j, char_b in enumerate(b, start=1):
            cost = 0 if char_a == char_b else 1
            table[i, j] = min(

                table[i - 1, j] + 1,
                table[i, j - 1] + 1,
                table[i - 1, j - 1] + cost,
            )

    return table[-1, -1]

# Compute a precomputed distance matrix.
n = len(objects)
distances = np.zeros((n, n), dtype=float)

for i, left in enumerate(objects):
    for j, right in enumerate(objects):
        distances[i, j] = edit_distance(left, right)

# Convert distances into similarity scores.
similarity = 1.0 / (1.0 + distances)
if distances.shape != (n, n):
    raise ValueError("Unexpected matrix shape.")

# Show the small precomputed matrices.
print("Objects:", objects)
print("Distance matrix:\n", distances)
print("Similarity matrix:\n", np.round(similarity, 2))

# Explain when precomputed input is appropriate.
print("Use precomputed input when pairwise comparison")
print("captures meaning better than ordinary columns.")
print("Examples include text, sequences, graphs,")
print("or expensive domain specific similarity rules.")



### **3.3. When to Precompute**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_A/image_03_03.jpg?v=1783837959" width="250">



>* Precompute costly similarities reused across experiments.
>* Best for specialized comparisons needing consistent results.

>* Use precomputed matrices for relationship-based data.
>* Reuse one similarity matrix across tasks.

>* Precompute carefully; storage and alignment add complexity.
>* Best for moderate, stable, costly comparisons.



In [ ]:
#@title Python Code - When to Precompute

# This topic is clearer with code.
# We compare reuse against recomputation.
# Small matrices keep the example readable.

import numpy as np
import time

# Use a fixed seed.
rng = np.random.default_rng(7)
# Create a tiny feature table.
X = rng.normal(size=(6, 3))

# Define an expensive distance rule.
def slow_distance(a, b):
    total = 0.0
    for _ in range(4000):
        total += np.sqrt(((a - b) ** 2).sum())
    return total / 4000

# Build one full distance matrix.
def pairwise_matrix(data):
    n_samples = data.shape[0]
    matrix = np.zeros((n_samples, n_samples))
    for i in range(n_samples):
        for j in range(n_samples):
            matrix[i, j] = slow_distance(data[i], data[j])
    return matrix

# Time repeated recomputation.
start_recompute = time.perf_counter()
for _ in range(2):
    recomputed = pairwise_matrix(X)
recompute_time = time.perf_counter() - start_recompute

# Time precompute plus reuse.
start_precompute = time.perf_counter()
precomputed = pairwise_matrix(X)
for _ in range(2):
    reused = precomputed.copy()
precompute_time = time.perf_counter() - start_precompute

# Validate the matrix shape.
assert precomputed.shape == (6, 6)
# Check symmetry for distances.
assert np.allclose(precomputed, precomputed.T)

# Simulate new incoming samples.
X_new = rng.normal(size=(2, 3))
# New data need fresh comparisons.
new_shape = (X_new.shape[0], X.shape[0])

# Print a compact teaching summary.
print("Distance matrix shape:", precomputed.shape)
print("Example distance:", round(precomputed[0, 1], 3))
print("Recompute twice seconds:", round(recompute_time, 3))
print("Precompute and reuse seconds:", round(precompute_time, 3))
print("Reuse is faster:", precompute_time < recompute_time)
print("Good when rule is costly and reused.")
print("Less ideal when new data arrive often.")
print("New-to-train matrix shape:", new_shape)



# <font color="#418FDE" size="6.5" uppercase>**Distances Kernels**</font>


In this lecture, you learned to:
- Compute and interpret pairwise distance matrices for common metrics. 
- Use kernel functions to represent similarity for linear and nonlinear modeling methods. 
- Decide when precomputed distances or kernels are appropriate in Scikit-Learn workflows. 

In the next Lecture (Lecture B), we will go over 'Kernel Approximation'